In [2]:
import torch
import pandas as pd

from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

from transformers import BertTokenizer, BertForSequenceClassification

from peft import PeftModel, PeftConfig

d:\coding\my-projects\fine-tuning-bert-on-fine-food\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
MODEL_NAME = "bert-base-uncased"
ADAPTER_PATH = "./bert-lora-adapter"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = BertTokenizer.from_pretrained(ADAPTER_PATH)

peft_config = PeftConfig.from_pretrained(ADAPTER_PATH)

In [4]:
base_model = BertForSequenceClassification.from_pretrained(
    peft_config.base_model_name_or_path,
    num_labels=2
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1703.97it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

In [5]:
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

In [6]:
df = pd.read_csv("data/balanced_reviews.csv")

_, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["labels"],
    random_state=42
)

texts = test_df["text"].tolist()
true_labels = test_df["labels"].tolist()

In [ ]:
from tqdm import tqdm

model.to(device)
model.eval()

predictions = []

with torch.no_grad():

    for text in tqdm(texts):

        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128,
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        outputs = model(**inputs)

        predicted_class = torch.argmax(
            outputs.logits,
            dim=-1
        ).item()

        predictions.append(predicted_class)


Running inference...


100%|██████████| 32815/32815 [07:44<00:00, 70.62it/s]


In [9]:
print(classification_report(true_labels,predictions))

              precision    recall  f1-score   support

           0       0.95      0.96      0.95     16408
           1       0.96      0.94      0.95     16407

    accuracy                           0.95     32815
   macro avg       0.95      0.95      0.95     32815
weighted avg       0.95      0.95      0.95     32815



# Test results

In [8]:
x_test, y_test = texts[:5], true_labels[:5]

In [10]:
for i in zip(x_test, y_test):
    print(i)

('This may be tasty -- but is incorrect to sell it as a "jam". It is (as the label\'s French text correctly says, a "Gelee" or jelly. Real red current jam or preserves is a whole different thing!', 0)
("I've tried many green teas in the same price range, but I always come back to Stash Organic Green.<br /><br />This green tea tastes how I think green tea should taste.  It's light, vegetal and very smooth.  To get the full flavor you need to use more leaves than most other green teas, and make sure the water temperature doesn't go over 180 degrees.  I often mix this tea with cascade mint, jasmine green or other Stash loose leaf herbal teas because although it's very good by itself, plain green tea can get boring day-after-day.  Contrary to the other reviewer, I don't like the Stash Organic Pinhead Gunpowder, and recommend this tea instead.", 1)
('My Cavalier Spaniel learned to sit within minutes. She is only 11 weeks old and will do anything for these treats!', 1)
('My fiance & I are fo

In [ ]:
model.to(device)
model.eval()

for text, label in zip(x_test, y_test):
    
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(device)
    
    outputs = model(**inputs)

    y_pred = torch.argmax(
        outputs.logits,
        dim=-1
    ).item()


    print("--------- Text -----------")
    print(text)
    print(f"Real label: {label}")
    print(f"Predicted label: {y_pred}")
    print("--------------------------")

--------- Text -----------
This may be tasty -- but is incorrect to sell it as a "jam". It is (as the label's French text correctly says, a "Gelee" or jelly. Real red current jam or preserves is a whole different thing!
Real label: 0
Predicted label: 0
--------------------------
--------- Text -----------
I've tried many green teas in the same price range, but I always come back to Stash Organic Green.<br /><br />This green tea tastes how I think green tea should taste.  It's light, vegetal and very smooth.  To get the full flavor you need to use more leaves than most other green teas, and make sure the water temperature doesn't go over 180 degrees.  I often mix this tea with cascade mint, jasmine green or other Stash loose leaf herbal teas because although it's very good by itself, plain green tea can get boring day-after-day.  Contrary to the other reviewer, I don't like the Stash Organic Pinhead Gunpowder, and recommend this tea instead.
Real label: 1
Predicted label: 1
------------

# Manual test

- 0 is Negative
- 1 is Positive

In [19]:
text = "This product may sound good, but its not what it says it to be. Its one of the scamy diet product just like everything else"
# text = "This product is something else, its so delicious"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
).to(device)
    
outputs = model(**inputs)

y_pred = torch.argmax(
    outputs.logits,
    dim=-1
).item()


print("--------- Text -----------")
print(text)
print(f"Predicted label: {y_pred}")
print("--------------------------")

--------- Text -----------
This product may sound good, but its not what it says it to be. Its one of the scamy diet product just like everything else
Predicted label: 0
--------------------------
